# FEATURE IMPORTANCE

### Return Variable 

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

# ----------------------------------------------------------------
# --- Load your data (adjust path/loading as needed) ---
# ----------------------------------------------------------------

X_raw = pd.read_csv("D://Joseph/Projects/kg-house-price-prediction/data/raw/test.csv")

In [ ]:
# ----------------------------------------------------------------
# ------- Load the model
# ----------------------------------------------------------------

MODEL_PATH = Path("D://Joseph/Projects/kg-house-price-prediction/artifacts/trained_models/xgboost_regression.joblib")
model = joblib.load(MODEL_PATH)
print(model)

# --- 1. Split the pipeline into preprocessing steps + regressor ---
preprocessing = model[:-1]  # everything before the regressor
regressor = model.named_steps["regressor"]  # the XGBRegressor itself

print(f"Print preprocessing {preprocessing}")

Pipeline(steps=[('feature_engineer', FeatureEngineer()),
                ('dropper', MissingRatioDropper(min_ratio=0.0)),
                ('preprocessor', Preprocessor()),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric='rm...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth

In [ ]:
# --- 2. Transform data through the preprocessing steps ---
X_transformed = preprocessing.transform(X_raw)

# Convert sparse -> dense before wrapping in a DataFrame
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

print("Print Data")
print(X_transformed)
print(type(X_transformed))
print(X_transformed.shape)

Print Data
[[ 1.71711532 -0.8667643   0.43004738 ...  0.          1.
   0.        ]
 [ 1.71946722 -0.8667643   0.47458348 ...  0.          1.
   0.        ]
 [ 1.72181913  0.07410996  0.16283075 ...  0.          1.
   0.        ]
 ...
 [ 5.1414913  -0.8667643   3.99293582 ...  0.          0.
   0.        ]
 [ 5.14384321  0.66215637 -0.37160252 ...  0.          1.
   0.        ]
 [ 5.14619511  0.07410996  0.16283075 ...  0.          1.
   0.        ]]
<class 'numpy.ndarray'>
(1459, 313)


In [ ]:
feature_names = preprocessing.get_feature_names_out()
X_transformed = pd.DataFrame(X_transformed, columns=feature_names)

print(f"\nTransformed shape: {X_transformed.shape}")
print(f"Number of feature names: {len(feature_names)}")


Transformed shape: (1459, 313)
Number of feature names: 313


In [ ]:
# --- 3. Built-in XGBoost gain importance ---
gain_importance = regressor.get_booster().get_score(importance_type="gain")
# XGBoost internally labels features f0, f1, ... — map back to real names
booster_feature_map = {f"f{i}": name for i, name in enumerate(feature_names)}
gain_series = pd.Series({booster_feature_map.get(k, k): v for k, v in gain_importance.items()})
gain_series = gain_series.sort_values(ascending=False)
print("\n=== Gain importance ===")
print(gain_series)

In [ ]:
gain_series.plot(kind="barh", figsize=(8, max(4, len(gain_series) * 0.3)))
plt.title("XGBoost Gain Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("gain_importance.png", dpi=150)
plt.close()

In [ ]:
# --- 4. SHAP global importance ---
explainer = shap.TreeExplainer(regressor)
shap_values = explainer.shap_values(X_transformed)

shap.summary_plot(shap_values, X_transformed, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("shap_bar.png", dpi=150)
plt.close()

In [ ]:
shap.summary_plot(shap_values, X_transformed, show=False)  # beeswarm
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150)
plt.close()

In [ ]:
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
print("\n=== Mean |SHAP value| ===")
print(mean_abs_shap)

# --- 5. Permutation importance (needs y_raw / y_true) ---
from sklearn.inspection import permutation_importance

# Permutation importance should run on the FULL pipeline against raw X, raw y,
# so preprocessing is re-applied correctly on each shuffle
perm = permutation_importance(model, X_raw, y_raw, n_repeats=10, random_state=0, scoring="neg_mean_squared_error")
perm_series = pd.Series(perm.importances_mean, index=X_raw.columns).sort_values(ascending=False)
print("\n=== Permutation importance (raw feature space) ===")
print(perm_series)

# --- 6. SHAP interaction values (optional, can be slow) ---
# Subsample if X_transformed is large
sample = X_transformed.sample(min(500, len(X_transformed)), random_state=0)
interaction_values = explainer.shap_interaction_values(sample)
mean_abs_interactions = np.abs(interaction_values).mean(axis=0)
interaction_df = pd.DataFrame(mean_abs_interactions, index=feature_names, columns=feature_names)
np.fill_diagonal(interaction_df.values, 0)
top_pairs = interaction_df.unstack().sort_values(ascending=False).drop_duplicates().head(10)
print("\n=== Top feature interactions ===")
print(top_pairs)